# 多元微积分三维可视化：VTK 版本

这个 notebook 专注于三维图形：二元函数曲面、曲面上的等高线，以及二次曲面的分类。二维梯度和切向量专题仍保留在原 notebook 中。

默认显示方式是 VTK 离屏渲染得到的静态 PNG，适合课堂演示和导出。若当前 Jupyter 环境支持 trame，也可以尝试交互式 VTK 视图。

## 0. 环境检查与导入

本 notebook 需要 `numpy`、`matplotlib`、`vtk`、`trame`、`trame_vtk` 和 `ipywidgets`。当前课程环境 `ai4math-vis` 的 `environment.yml` 已包含这些包。

In [ ]:
import importlib.util
import os
import sys

os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib")

required_packages = ["numpy", "matplotlib", "vtk", "trame", "trame_vtk", "ipywidgets"]
missing_packages = [name for name in required_packages if importlib.util.find_spec(name) is None]

print(f"Python: {sys.version.split()[0]}")
if missing_packages:
    print("缺少依赖：" + ", ".join(missing_packages))
    print("请先激活课程环境：conda activate ai4math-vis")
    raise ModuleNotFoundError("Missing packages: " + ", ".join(missing_packages))

import numpy as np
import matplotlib.pyplot as plt
import vtk
vtk.vtkObject.GlobalWarningDisplayOff()
vtk.vtkLogger.SetStderrVerbosity(vtk.vtkLogger.VERBOSITY_ERROR)
from IPython.display import Image, display
from ipywidgets import FloatSlider, interact
from matplotlib import font_manager

chinese_font_candidates = [
    "Noto Sans CJK SC",
    "Noto Sans CJK JP",
    "Noto Sans CJK TC",
    "Source Han Sans SC",
    "Source Han Sans CN",
    "WenQuanYi Micro Hei",
    "Microsoft YaHei",
    "SimHei",
    "Arial Unicode MS",
]
available_font_names = {font.name for font in font_manager.fontManager.ttflist}
matplotlib_chinese_font = next((name for name in chinese_font_candidates if name in available_font_names), None)
plt.rcParams["font.family"] = "sans-serif"
if matplotlib_chinese_font:
    plt.rcParams["font.sans-serif"] = [matplotlib_chinese_font, "DejaVu Sans"]
else:
    plt.rcParams["font.sans-serif"] = chinese_font_candidates + ["DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

print("依赖检查通过。")
vtk_chinese_font_file = None
if matplotlib_chinese_font:
    try:
        vtk_chinese_font_file = font_manager.findfont(matplotlib_chinese_font, fallback_to_default=False)
    except Exception:
        vtk_chinese_font_file = None
    print("Matplotlib 中文字体：" + matplotlib_chinese_font)


## 0.1 VTK 管线速查

- `vtkStructuredGrid`：保存规则网格上的三维曲面点。
- `vtkDataSetMapper`：把曲面数据映射成可渲染对象，并按函数值着色。
- `vtkContourFilter`：从曲面标量场中提取等高线。
- `vtkLookupTable` / `vtkScalarBarActor`：控制颜色映射和色条。
- `vtkWindowToImageFilter`：把 VTK 渲染窗口转成 PNG，嵌入 notebook。

## 1. 通用 VTK 绘图工具

In [ ]:
def make_grid(xlim=(-3, 3), ylim=(-3, 3), n=121):
    """生成二维采样网格。"""
    x = np.linspace(xlim[0], xlim[1], n)
    y = np.linspace(ylim[0], ylim[1], n)
    X, Y = np.meshgrid(x, y)
    return x, y, X, Y


def evaluate_on_grid(f, X, Y):
    """在网格上计算函数值，并要求返回值与网格同形状。"""
    Z = np.asarray(f(X, Y), dtype=float)
    if Z.shape != X.shape:
        raise ValueError("函数必须能接受 numpy 数组，并返回与网格同形状的数组。")
    return Z


def make_vtk_surface(X, Y, Z):
    """把 NumPy 网格转换成带标量值的 vtkStructuredGrid。"""
    ny, nx = Z.shape
    points = vtk.vtkPoints()
    scalars = vtk.vtkDoubleArray()
    scalars.SetName("f(x,y)")

    for j in range(ny):
        for i in range(nx):
            z_value = float(Z[j, i])
            points.InsertNextPoint(float(X[j, i]), float(Y[j, i]), z_value)
            scalars.InsertNextValue(z_value)

    grid = vtk.vtkStructuredGrid()
    grid.SetDimensions(nx, ny, 1)
    grid.SetPoints(points)
    grid.GetPointData().SetScalars(scalars)
    return grid


def make_lookup_table(zmin, zmax):
    """创建近似 Viridis 风格的查找表。"""
    lut = vtk.vtkLookupTable()
    lut.SetNumberOfTableValues(256)
    lut.SetRange(float(zmin), float(zmax))
    lut.SetHueRange(0.72, 0.15)
    lut.SetSaturationRange(0.85, 0.95)
    lut.SetValueRange(0.45, 0.95)
    lut.Build()
    return lut


def make_text_actor(text, position=(18, 570), font_size=22):
    actor = vtk.vtkTextActor()
    actor.SetInput(text)
    actor.SetPosition(*position)
    prop = actor.GetTextProperty()
    prop.SetFontSize(font_size)
    prop.SetColor(0.08, 0.08, 0.08)
    if vtk_chinese_font_file:
        prop.SetFontFamily(vtk.VTK_FONT_FILE)
        prop.SetFontFile(vtk_chinese_font_file)
    else:
        prop.SetFontFamilyToArial()
    return actor


def make_vtk_surface_renderer(X, Y, Z, title="", width=900, height=650, contour_count=13):
    """创建包含曲面、三维等高线、坐标轴和色条的 VTK renderer。"""
    grid = make_vtk_surface(X, Y, Z)
    zmin, zmax = float(np.nanmin(Z)), float(np.nanmax(Z))
    lut = make_lookup_table(zmin, zmax)

    surface_mapper = vtk.vtkDataSetMapper()
    surface_mapper.SetInputData(grid)
    surface_mapper.SetLookupTable(lut)
    surface_mapper.SetScalarRange(zmin, zmax)

    surface_actor = vtk.vtkActor()
    surface_actor.SetMapper(surface_mapper)
    surface_actor.GetProperty().SetInterpolationToPhong()

    contour = vtk.vtkContourFilter()
    contour.SetInputData(grid)
    if np.isclose(zmin, zmax):
        contour.SetValue(0, zmin)
    else:
        contour.GenerateValues(contour_count, zmin, zmax)

    contour_mapper = vtk.vtkPolyDataMapper()
    contour_mapper.SetInputConnection(contour.GetOutputPort())
    contour_mapper.SetLookupTable(lut)
    contour_mapper.SetScalarRange(zmin, zmax)

    contour_actor = vtk.vtkActor()
    contour_actor.SetMapper(contour_mapper)
    contour_actor.GetProperty().SetColor(1.0, 1.0, 1.0)
    contour_actor.GetProperty().SetLineWidth(2.0)

    scalar_bar = vtk.vtkScalarBarActor()
    scalar_bar.SetLookupTable(lut)
    scalar_bar.SetTitle("f(x,y)")
    scalar_bar.SetNumberOfLabels(5)
    scalar_bar.SetMaximumWidthInPixels(80)
    scalar_bar.SetMaximumHeightInPixels(420)
    scalar_bar.GetTitleTextProperty().SetColor(0.05, 0.05, 0.05)
    scalar_bar.GetLabelTextProperty().SetColor(0.05, 0.05, 0.05)

    axes = vtk.vtkCubeAxesActor()
    axes.SetBounds(grid.GetBounds())
    axes.SetXTitle("x")
    axes.SetYTitle("y")
    axes.SetZTitle("z")
    axes.DrawXGridlinesOn()
    axes.DrawYGridlinesOn()
    axes.DrawZGridlinesOn()

    renderer = vtk.vtkRenderer()
    renderer.SetBackground(1.0, 1.0, 1.0)
    renderer.AddActor(surface_actor)
    renderer.AddActor(contour_actor)
    renderer.AddActor2D(scalar_bar)
    if title:
        renderer.AddActor2D(make_text_actor(title, position=(18, height - 42)))

    camera = renderer.GetActiveCamera()
    axes.SetCamera(camera)
    renderer.AddActor(axes)

    renderer.ResetCamera()
    camera.Azimuth(35)
    camera.Elevation(28)
    camera.Zoom(1.08)
    renderer.ResetCameraClippingRange()

    render_window = vtk.vtkRenderWindow()
    render_window.SetSize(width, height)
    render_window.AddRenderer(renderer)
    return render_window, renderer


def render_vtk_png(render_window):
    """离屏渲染 VTK 窗口，并在 notebook 中显示 PNG。"""
    render_window.OffScreenRenderingOn()
    render_window.Render()

    window_to_image = vtk.vtkWindowToImageFilter()
    window_to_image.SetInput(render_window)
    window_to_image.SetScale(1)
    window_to_image.SetInputBufferTypeToRGBA()
    window_to_image.ReadFrontBufferOff()
    window_to_image.Update()

    writer = vtk.vtkPNGWriter()
    writer.SetWriteToMemory(True)
    writer.SetInputConnection(window_to_image.GetOutputPort())
    writer.Write()
    png_bytes = memoryview(writer.GetResult()).tobytes()
    display(Image(data=png_bytes))
    return png_bytes


def show_vtk_interactive(render_window):
    """尝试用 trame 在 notebook 中显示可交互 VTK 视图。失败时返回 False。"""
    try:
        from trame.app import get_server
        from trame.ui.vuetify3 import SinglePageLayout
        from trame.widgets import vtk as vtk_widgets

        server = get_server(client_type="vue3")
        with SinglePageLayout(server) as layout:
            layout.title.set_text("VTK 交互视图")
            with layout.content:
                view = vtk_widgets.VtkRemoteView(render_window)
                server.controller.view_update = view.update
        display(layout)
        return True
    except Exception as exc:
        print("trame 交互视图不可用，已回退到静态 PNG。原因：", exc)
        return False


def plot_vtk_surface(f, xlim=(-3, 3), ylim=(-3, 3), n=121, title="", interactive=False):
    """用 VTK 绘制 z=f(x,y) 曲面，并在曲面上叠加三维等高线。"""
    _, _, X, Y = make_grid(xlim, ylim, n)
    Z = evaluate_on_grid(f, X, Y)
    render_window, renderer = make_vtk_surface_renderer(X, Y, Z, title=title)
    if interactive and show_vtk_interactive(render_window):
        return render_window, renderer
    render_vtk_png(render_window)
    return render_window, renderer


## 2. 二元函数曲面示例

In [ ]:
def bowl(x, y):
    return x**2 + y**2


plot_vtk_surface(bowl, xlim=(-3, 3), ylim=(-3, 3), title="例 1：f(x, y) = x² + y²")

In [ ]:
def saddle(x, y):
    return x**2 - y**2


plot_vtk_surface(saddle, xlim=(-3, 3), ylim=(-3, 3), title="例 2：f(x, y) = x² - y²")

In [ ]:
def wave(x, y):
    return np.sin(x) * np.cos(y)


plot_vtk_surface(wave, xlim=(-2 * np.pi, 2 * np.pi), ylim=(-2 * np.pi, 2 * np.pi), title="例 3：f(x, y) = sin(x)cos(y)")

## 3. 二次曲面：用二次型矩阵分类

考虑

$$f(x,y)=\begin{bmatrix}x & y\end{bmatrix}A\begin{bmatrix}x\\y\end{bmatrix}+b^T\begin{bmatrix}x\\y\end{bmatrix}+c.$$

矩阵 $A$ 的特征值符号决定曲面的主要形状。

In [ ]:
def symmetrize(A):
    A = np.asarray(A, dtype=float)
    if A.shape != (2, 2):
        raise ValueError("A 必须是 2x2 矩阵。")
    return 0.5 * (A + A.T)


def classify_quadratic(A, tol=1e-10):
    """按二次型矩阵特征值符号给出二次曲面分类。"""
    A = symmetrize(A)
    eigenvalues = np.linalg.eigvalsh(A)
    positive = eigenvalues > tol
    negative = eigenvalues < -tol
    zero = np.abs(eigenvalues) <= tol

    if positive.all():
        classification = "正定：椭圆抛物面，开口向上"
    elif negative.all():
        classification = "负定：倒椭圆抛物面，开口向下"
    elif positive.any() and negative.any():
        classification = "不定：双曲抛物面，也叫鞍面"
    elif positive.any() and zero.any():
        classification = "半正定：抛物柱面或退化椭圆抛物面"
    elif negative.any() and zero.any():
        classification = "半负定：倒抛物柱面或退化椭圆抛物面"
    else:
        classification = "零二次项：平面或常数面"

    return {"A": A, "eigenvalues": eigenvalues, "classification": classification}


def quadratic_function(A, b=(0, 0), c=0):
    """返回 f(x,y)= [x,y]A[x,y]^T + b^T[x,y] + c。"""
    A = symmetrize(A)
    b = np.asarray(b, dtype=float)
    if b.shape != (2,):
        raise ValueError("b 必须是长度为 2 的向量。")

    def f(x, y):
        return A[0, 0] * x**2 + 2 * A[0, 1] * x * y + A[1, 1] * y**2 + b[0] * x + b[1] * y + c

    return f


def plot_quadratic_surface_vtk(A, b=(0, 0), c=0, xlim=(-3, 3), ylim=(-3, 3), n=121, title=None, interactive=False):
    info = classify_quadratic(A)
    f = quadratic_function(info["A"], b=b, c=c)
    eigen_text = np.array2string(info["eigenvalues"], precision=3)
    display_title = title or info["classification"]
    print("A =")
    print(info["A"])
    print(f"特征值：{eigen_text}")
    print(f"分类：{info['classification']}")
    plot_vtk_surface(f, xlim=xlim, ylim=ylim, n=n, title=display_title, interactive=interactive)
    return info


## 4. 二次曲面示例

In [ ]:
A_positive = np.array([[1, 0], [0, 1]])
plot_quadratic_surface_vtk(A_positive, title="正定矩阵：f(x, y) = x² + y²");

In [ ]:
A_negative = np.array([[-1, 0], [0, -1]])
plot_quadratic_surface_vtk(A_negative, title="负定矩阵：f(x, y) = -x² - y²");

In [ ]:
A_indefinite = np.array([[1, 0], [0, -1]])
plot_quadratic_surface_vtk(A_indefinite, title="不定矩阵：f(x, y) = x² - y²");

In [ ]:
A_semipositive = np.array([[1, 0], [0, 0]])
plot_quadratic_surface_vtk(A_semipositive, title="半正定矩阵：f(x, y) = x²");

In [ ]:
plot_quadratic_surface_vtk(
    A_positive,
    b=(-2, 1),
    c=0,
    xlim=(-1, 4),
    ylim=(-4, 2),
    title="线性项移动顶点：x²+y²-2x+y",
);

## 5. 交互实验：调节二次型矩阵

默认仍使用静态 PNG，滑块改变后重新渲染一次 VTK 曲面。

In [ ]:
def explore_quadratic_vtk(a=1.0, h=0.0, d=1.0):
    A = np.array([[a, h], [h, d]], dtype=float)
    info = classify_quadratic(A)
    eigen_text = np.array2string(info["eigenvalues"], precision=3)
    print(f"A = [[{a:.2f}, {h:.2f}], [{h:.2f}, {d:.2f}]]")
    print(f"特征值：{eigen_text}")
    print(f"分类：{info['classification']}")
    f = quadratic_function(A)
    plot_vtk_surface(f, xlim=(-3, 3), ylim=(-3, 3), n=81, title="交互式二次曲面")


interact(
    explore_quadratic_vtk,
    a=FloatSlider(value=1.0, min=-2.0, max=2.0, step=0.25, description="a, 第一对角元"),
    h=FloatSlider(value=0.0, min=-2.0, max=2.0, step=0.25, description="h, 非对角元"),
    d=FloatSlider(value=1.0, min=-2.0, max=2.0, step=0.25, description="d, 第二对角元"),
);

## 6. 可选：尝试交互式 VTK 视图

如果当前 JupyterLab 和 trame 后端支持，下面的单元会显示可旋转缩放的 VTK 视图；否则会自动回退到静态 PNG。

In [ ]:
# 将 interactive 改为 True 可以尝试 trame 交互视图。
plot_vtk_surface(bowl, xlim=(-3, 3), ylim=(-3, 3), n=81, title="可选交互视图：x² + y²", interactive=True)